<div style="background: linear-gradient(135deg, #1e50a0 0%, #2e6cc0 100%); padding: 36px 40px; border-radius: 10px; color: white; margin-bottom: 20px;">
    <h1 style="color: white; margin: 0 0 8px 0; font-size: 2em;">Introduction to PINNs | Dynamics Case</h1>
    <h2 style="color: #c8dcff; margin: 0 0 12px 0; font-weight: normal; font-size: 1.3em;">Double Pendulum: a 2-DOF System of Coupled Nonlinear ODEs</h2>
    <p style="color: #a0bce0; margin: 0; font-size: 1em;">Hybrid PINN &nbsp;·&nbsp; Initial conditions + physics + noisy data &nbsp;·&nbsp; Multi-output network &nbsp;·&nbsp; Normalisation &nbsp;·&nbsp; PyTorch</p>
    <hr>
    <p style="color: #a0bce0; margin: 0; font-size: 1em;">Author: A.S.Grm (aleksander.grm@fpp.uni-lj.si), 2026</p>
</div>

### Problem summary

Two rigid arms (lengths $L_1, L_2$, point masses $m_1, m_2$) swing in a vertical plane without friction.
With $\delta = \theta_2 - \theta_1$, the Lagrangian equations of motion are

$$r_1 = (m_1+m_2)L_1\ddot\theta_1 + m_2L_2\ddot\theta_2\cos\delta - m_2L_2\dot\theta_2^2\sin\delta + (m_1+m_2)g\sin\theta_1 = 0$$
$$r_2 = L_2\ddot\theta_2 + L_1\ddot\theta_1\cos\delta + L_1\dot\theta_1^2\sin\delta + g\sin\theta_2 = 0$$

Initial conditions: $\theta_1(0)=\pi/4,\ \theta_2(0)=0,\ \dot\theta_1(0)=\dot\theta_2(0)=0$; time window $t\in[0,5]$ s; $g=9.81$ m/s², $L_1=L_2=1$ m, $m_1=m_2=1$ kg.

**PINN:** one network $\tilde t \mapsto (\tilde\theta_1, \tilde\theta_2)$ (normalised in and out), trained with

$$\mathcal L = \lambda_\text{IC}\,\mathcal L_\text{IC} + \lambda_\text{phys}\,\frac{1}{N_f}\sum_i\big(r_1^2 + r_2^2\big) + \lambda_\text{data}\,\frac{1}{N_d}\sum_j\big\|\tilde{\boldsymbol\theta}(\tilde t_j) - \tilde{\boldsymbol\theta}^\text{exp}_j\big\|^2 .$$

Validation: comparison with a high-accuracy numerical reference solution (`scipy.integrate.solve_ivp`, DOP853).

| Code cell | Content |
|---|---|
| 1–2 | imports, compute device |
| 3–5 | physical parameters, reference solution, synthetic noisy measurements |
| 6–7 | normaliser, neural network |
| 8–9 | ODE residuals (autograd), collocation points |
| 10 | training loop |
| 11–12 | experiment driver with validation metrics, results dashboard |
| 13–14 | `main()` and entry point |

In [ ]:
import time
import platform
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from tqdm.auto import tqdm

In [ ]:
# Prefer GPU (CUDA / AMD ROCm / Apple Silicon); fall back to CPU when no GPU is detected.
# dd = None  -> automatic selection
# dd = 'cpu' | 'cuda' | 'mps' -> force a specific device
def get_device(dd=None):

    if dd is None:
        if torch.cuda.is_available():
            device = torch.device("cuda")
            device_name = torch.cuda.get_device_name(0)
        elif torch.backends.mps.is_available():
            device = torch.device("mps")
            device_name = f"Apple Silicon ({platform.processor()})"
        else:
            device = torch.device("cpu")
            device_name = "GPU not detected – training will be slower"
    else:
        if dd == "cuda":
            device = torch.device("cuda")
            device_name = torch.cuda.get_device_name(0)
        elif dd == "mps":
            device = torch.device("mps")
            device_name = f"Apple Silicon ({platform.processor()})"
        elif dd == "cpu":
            device = torch.device("cpu")
            device_name = "CPU (forced)"
        else:
            raise ValueError(f"Device name is not supported: {dd}")

    print(f"Compute device : {device} ({device_name})")

    return device

In [ ]:
# A dataclass is a compact container for named values.
# Collecting all physical constants in ONE object avoids hidden global variables:
# every function that needs a parameter receives 'p' explicitly.
@dataclass
class DoublePendulumParams:

    g:  float = 9.81              # gravitational acceleration [m/s^2]
    L1: float = 1.0               # length of the top arm [m]
    L2: float = 1.0               # length of the bottom arm [m]
    m1: float = 1.0               # mass of the top bob [kg]
    m2: float = 1.0               # mass of the bottom bob [kg]

    # Initial state at t = 0: angles [rad] and angular velocities [rad/s]
    theta1_0: float = np.pi / 4   # 45° top arm
    theta2_0: float = 0.0         # bottom arm hanging straight down
    omega1_0: float = 0.0         # released from rest
    omega2_0: float = 0.0

    # Time window of the simulation [s]
    t_start: float = 0.0
    t_end:   float = 5.0

    # Convenience property: the initial state as a list [theta1, theta2, omega1, omega2]
    @property
    def initial_state(self):
        return [self.theta1_0, self.theta2_0, self.omega1_0, self.omega2_0]

In [ ]:
def double_pendulum_rhs(t, state, p):
    """
    Right-hand side of the double pendulum written as a FIRST-ORDER system
    ds/dt = F(s), with state s = [theta1, theta2, omega1, omega2].

    The two coupled equations of motion (see problem summary) are linear in the
    accelerations (theta1'', theta2''). Solving that 2x2 linear system explicitly
    gives the expressions for d(omega1)/dt and d(omega2)/dt below.

    Parameters
    ----------
    t : float
        Time [s] (unused – the system is autonomous – but required by solve_ivp).
    state : array_like, shape (4,)
        Current state [theta1, theta2, omega1, omega2].
    p : DoublePendulumParams
        Physical parameters.

    Returns
    -------
    list of float
        Time derivatives [theta1', theta2', omega1', omega2'].
    """
    th1, th2, w1, w2 = state
    g, L1, L2, m1, m2 = p.g, p.L1, p.L2, p.m1, p.m2

    delta = th2 - th1                          # relative angle between the arms

    # Common denominator (determinant of the 2x2 mass matrix, scaled)
    den1 = (m1 + m2) * L1 - m2 * L1 * np.cos(delta)**2
    den2 = (L2 / L1) * den1

    # Angular acceleration of the top arm
    dw1 = (m2 * L1 * w1**2 * np.sin(delta) * np.cos(delta)
           + m2 * g * np.sin(th2) * np.cos(delta)
           + m2 * L2 * w2**2 * np.sin(delta)
           - (m1 + m2) * g * np.sin(th1)) / den1

    # Angular acceleration of the bottom arm
    dw2 = (-m2 * L2 * w2**2 * np.sin(delta) * np.cos(delta)
           + (m1 + m2) * g * np.sin(th1) * np.cos(delta)
           - (m1 + m2) * L1 * w1**2 * np.sin(delta)
           - (m1 + m2) * g * np.sin(th2)) / den2

    # d(theta)/dt = omega ; d(omega)/dt = angular acceleration
    return [w1, w2, dw1, dw2]


def solve_reference(p, t_eval):
    """
    High-accuracy numerical reference solution used for (i) generating synthetic
    measurements and (ii) validating the trained PINN.

    DOP853 is an explicit 8th-order Runge–Kutta method; with tight tolerances the
    error is many orders of magnitude below anything the PINN can reach.

    Returns
    -------
    sol : np.ndarray, shape (len(t_eval), 4)
        Columns: theta1, theta2, omega1, omega2.
    """
    out = solve_ivp(
        fun=double_pendulum_rhs,
        t_span=(p.t_start, p.t_end),
        y0=p.initial_state,
        t_eval=t_eval,
        args=(p,),               # extra arguments forwarded to double_pendulum_rhs
        method="DOP853",
        rtol=1e-10,
        atol=1e-12,
    )
    if not out.success:
        raise RuntimeError(f"Reference ODE solver failed: {out.message}")

    # solve_ivp returns shape (4, N); transpose to (N, 4) = one row per time instant
    return out.y.T

In [ ]:
def generate_experimental_data(p, N_data, noise_std, seed):
    """
    Creates synthetic "measurements": both angles sampled at the SAME random
    time instants (two sensors, one on each arm) and corrupted by Gaussian noise.

    Parameters
    ----------
    p : DoublePendulumParams
    N_data : int
        Number of measurement instants.
    noise_std : float
        Standard deviation of the additive sensor noise [rad].
    seed : int
        Seed for NumPy's random generator (reproducible data).

    Returns
    -------
    t_data : np.ndarray, shape (N_data,)
        Sorted measurement times [s].
    theta_data : np.ndarray, shape (N_data, 2)
        Noisy angles; column 0 = theta1, column 1 = theta2 [rad].
    """
    # Independent, reproducible random generator (does not disturb global NumPy state)
    rng = np.random.default_rng(seed)

    # Random, sorted time instants inside the simulation window
    t_data = np.sort(rng.uniform(p.t_start, p.t_end, N_data))

    # Exact angles at those instants, taken from the reference solution
    theta_exact = solve_reference(p, t_data)[:, 0:2]

    # Add zero-mean Gaussian noise to each angle independently
    theta_data = theta_exact + noise_std * rng.standard_normal(theta_exact.shape)

    return t_data, theta_data

In [ ]:
# Normalisation keeps the network's inputs and outputs of order one.
# tanh networks train best when inputs lie roughly in [-1, 1] and targets have
# zero mean / unit variance; raw seconds and radians are NOT on that scale.
class Normaliser:

    def __init__(self, t_min, t_max, theta_data, device):

        # ---- time: affine map [t_min, t_max] -> [-1, 1] -------------------
        self.t_min = t_min
        self.t_max = t_max

        # ---- angles: standardisation, SEPARATE statistics for each angle --
        # theta1 and theta2 have different means and amplitudes; using one
        # common scale would let the larger-amplitude angle dominate the loss.
        # Shape (1, 2) so that it broadcasts over a batch of shape (N, 2).
        mu    = theta_data.mean(axis=0, keepdims=True)
        sigma = theta_data.std(axis=0, keepdims=True)
        self.mu    = torch.tensor(mu,    dtype=torch.float32, device=device)
        self.sigma = torch.tensor(sigma, dtype=torch.float32, device=device)

    # Physical time t [s]  ->  normalised time t~ in [-1, 1]
    def normalise_t(self, t):
        return 2.0 * (t - self.t_min) / (self.t_max - self.t_min) - 1.0

    # Physical angles [rad]  ->  normalised angles (zero mean, unit std)
    def normalise_theta(self, theta):
        return (theta - self.mu) / self.sigma

    # Normalised network output  ->  physical angles [rad]
    def denormalise_theta(self, theta_norm):
        return theta_norm * self.sigma + self.mu

In [ ]:
# Define a custom neural network class that inherits from PyTorch's base model class (nn.Module).
# Inheriting from nn.Module gives us built-in tracking of parameters, weights, and gradients.
class PINN2DOF(nn.Module):

    # The constructor builds the architecture:
    #   1 input  (normalised time t~)
    #   num_layers hidden layers with hidden_dim neurons each, tanh activation
    #   2 outputs (normalised theta1~, theta2~)
    def __init__(self, hidden_dim, num_layers):

        # Required call: initialises PyTorch's internal parameter bookkeeping.
        super().__init__()

        # FIRST LAYER: 1 input -> hidden_dim neurons.
        # tanh is smooth and infinitely differentiable; we need SECOND derivatives
        # of the output w.r.t. time (accelerations), so ReLU (zero curvature) is unsuitable.
        layers = [nn.Linear(1, hidden_dim), nn.Tanh()]

        # HIDDEN LAYERS: (num_layers - 1) additional blocks hidden_dim -> hidden_dim.
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())

        # OUTPUT LAYER: hidden_dim -> 2 neurons, linear (no activation),
        # because the targets are unbounded real numbers.
        # ONE network with TWO outputs shares its hidden features between both arms,
        # which is natural because the two angles are physically coupled.
        layers.append(nn.Linear(hidden_dim, 2))

        # Chain all layers into a single callable pipeline.
        self.net = nn.Sequential(*layers)

    # forward(): how an input batch of shape (N, 1) is mapped to an output of shape (N, 2).
    # PyTorch calls it automatically when we write model(t_norm).
    def forward(self, t_norm):
        return self.net(t_norm)

In [ ]:
def predict_theta(model, t, norm):
    """
    Full prediction chain in PHYSICAL units:
        t [s] --normalise--> t~ --network--> theta~ --denormalise--> theta [rad]

    Because normalisation and denormalisation are ordinary torch operations,
    autograd differentiates THROUGH them. Derivatives taken w.r.t. 't' are therefore
    automatically in physical units (rad/s, rad/s^2) – the chain rule factors
    2/(t_max - t_min) and sigma are included without any manual bookkeeping.

    Returns
    -------
    theta : torch.Tensor, shape (N, 2)   physical angles [rad]
    """
    return norm.denormalise_theta(model(norm.normalise_t(t)))


def compute_ode_residual(model, t, norm, p):
    """
    Evaluates the residuals of the two coupled equations of motion at time points t.

    Parameters
    ----------
    model : nn.Module
        PINN2DOF network.
    t : torch.Tensor, shape (N, 1), requires_grad=True
        Physical collocation times [s].
    norm : Normaliser
    p : DoublePendulumParams

    Returns
    -------
    r1, r2 : torch.Tensor, shape (N, 1)
        Residuals of equation 1 (top arm) and equation 2 (bottom arm). Target: 0.
    """
    # ---- Forward pass: angles of both arms in physical units ------------
    theta = predict_theta(model, t, norm)

    # Slice the two output columns. '0:1' (not '0') keeps shape (N, 1),
    # which is what autograd.grad expects to match the input tensor t.
    th1 = theta[:, 0:1]
    th2 = theta[:, 1:2]

    # ---- Helper: derivative of a column tensor y(t) w.r.t. t ------------
    # grad_outputs = ones: each output depends only on "its own" time point,
    #                so summing with weight 1 returns the pointwise derivative.
    # create_graph = True: the derivative itself stays differentiable, which is
    #                required (a) for the second derivative and (b) so that the
    #                loss built from these derivatives can be back-propagated
    #                to the network weights.
    def d_dt(y):
        return torch.autograd.grad(
            outputs=y,
            inputs=t,
            grad_outputs=torch.ones_like(y),
            create_graph=True,
        )[0]

    # ---- First derivatives: angular velocities [rad/s] ------------------
    dth1 = d_dt(th1)
    dth2 = d_dt(th2)

    # ---- Second derivatives: angular accelerations [rad/s^2] ------------
    d2th1 = d_dt(dth1)
    d2th2 = d_dt(dth2)

    # ---- Residuals of the equations of motion ---------------------------
    g, L1, L2, m1, m2 = p.g, p.L1, p.L2, p.m1, p.m2
    delta = th2 - th1

    # Equation 1 (top arm): includes the inertial coupling term m2*L2*th2''*cos(delta)
    r1 = ((m1 + m2) * L1 * d2th1
          + m2 * L2 * d2th2 * torch.cos(delta)
          - m2 * L2 * dth2**2 * torch.sin(delta)
          + (m1 + m2) * g * torch.sin(th1))

    # Equation 2 (bottom arm): includes the coupling term L1*th1''*cos(delta)
    r2 = (L2 * d2th2
          + L1 * d2th1 * torch.cos(delta)
          + L1 * dth1**2 * torch.sin(delta)
          + g * torch.sin(th2))

    # If the network predicted the exact motion, r1 = r2 = 0 at every point.
    return r1, r2

In [ ]:
def generate_domain_points(t_start, t_end, N_f, device):
    """
    Builds the time points on which the different loss terms are evaluated.

    Returns
    -------
    t_f : torch.Tensor, shape (N_f, 1), requires_grad=True
        Collocation points in [t_start, t_end] for the physics residual.
        No measured values are needed here – the ODE itself is the "label".
    t_ic : torch.Tensor, shape (1, 1), requires_grad=True
        The initial time t = t_start, where angles AND velocities are prescribed.
    """
    # Uniformly spaced collocation points; .unsqueeze(1): shape [N_f] -> [N_f, 1]
    t_f = torch.linspace(t_start, t_end, N_f, device=device).unsqueeze(1)

    # requires_grad = True: autograd must track t_f to compute d(theta)/dt and d2(theta)/dt2.
    t_f.requires_grad_(True)

    # Initial time point; gradients are needed because the initial VELOCITIES
    # (d(theta)/dt at t = 0) are part of the initial-condition loss.
    t_ic = torch.tensor([[t_start]], device=device, requires_grad=True)

    return t_f, t_ic

In [ ]:
def train_pinn(
    model,
    norm,
    p,
    t_f,
    t_ic,
    t_data,
    theta_data,
    epochs,
    lr,
    lambda_ic,
    lambda_phys,
    lambda_data,
    print_every,
):
    """
    Trains the double-pendulum PINN with Adam and a step learning-rate schedule.

    Parameters
    ----------
    model : PINN2DOF
    norm : Normaliser
    p : DoublePendulumParams
    t_f, t_ic : torch.Tensor
        Collocation points and initial time (see generate_domain_points).
    t_data : torch.Tensor, shape (N_d, 1)
        Measurement times [s].
    theta_data : torch.Tensor, shape (N_d, 2)
        Noisy measured angles [rad].
    epochs : int
        Number of optimisation steps.
    lr : float
        Initial Adam learning rate.
    lambda_ic, lambda_phys, lambda_data : float
        Weights of the three loss terms. lambda_data = 0 gives a pure-physics PINN.
    print_every : int
        Logging interval in epochs.

    Returns
    -------
    history : dict of lists
        Per-epoch values of 'total', 'ic', 'phys', 'data' losses and 'lr'.
    """
    # Adam: adaptive gradient descent – robust default for PINNs.
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Halve the learning rate four times during training (staircase decay).
    # Large steps early for fast progress, small steps late for fine convergence.
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=max(1, epochs // 4), gamma=0.5
    )

    # The data loss is computed in NORMALISED space: both angles then have unit
    # variance and contribute equally. These targets never change -> precompute.
    t_data_norm     = norm.normalise_t(t_data)
    theta_data_norm = norm.normalise_theta(theta_data)

    # Prescribed initial state as tensors of shape (1, 2)
    theta_0 = torch.tensor([[p.theta1_0, p.theta2_0]], device=t_f.device)
    omega_0 = torch.tensor([[p.omega1_0, p.omega2_0]], device=t_f.device)

    history = {"total": [], "ic": [], "phys": [], "data": [], "lr": []}

    for epoch in tqdm(range(epochs + 1), desc="Training", unit="epoch", mininterval=1.0):

        # CLEAR GRADIENTS: PyTorch accumulates gradients across backward() calls.
        optimizer.zero_grad()

        # =================================================================
        # TERM 1: Initial-condition loss  (angles AND velocities at t = 0)
        # =================================================================
        theta_ic = predict_theta(model, t_ic, norm)                  # shape (1, 2)

        # Angular velocities at t = 0 – one autograd call PER output column,
        # because each column must be differentiated separately.
        omega_ic = torch.cat([
            torch.autograd.grad(theta_ic[:, k:k+1], t_ic,
                                torch.ones_like(theta_ic[:, k:k+1]),
                                create_graph=True)[0]
            for k in range(2)
        ], dim=1)                                                    # shape (1, 2)

        loss_ic = (torch.mean((theta_ic - theta_0)**2)               # angle errors
                   + torch.mean((omega_ic - omega_0)**2))            # velocity errors

        # =================================================================
        # TERM 2: Physics loss  (both coupled ODE residuals)
        # =================================================================
        r1, r2 = compute_ode_residual(model, t_f, norm, p)

        # BOTH residuals must vanish; enforcing only r1 would leave the
        # bottom-arm motion physically unconstrained.
        loss_phys = torch.mean(r1**2) + torch.mean(r2**2)

        # =================================================================
        # TERM 3: Data loss  (noisy measurements, normalised space)
        # =================================================================
        loss_data = torch.mean((model(t_data_norm) - theta_data_norm)**2)

        # =================================================================
        # TOTAL composite loss
        # =================================================================
        loss = lambda_ic * loss_ic + lambda_phys * loss_phys + lambda_data * loss_data

        # BACKWARD PASS: gradients of the loss w.r.t. all weights and biases.
        loss.backward()

        # OPTIMISER STEP: update the parameters, then advance the lr schedule.
        optimizer.step()
        scheduler.step()

        # RECORD: .item() converts a one-element tensor into a Python float.
        history["total"].append(loss.item())
        history["ic"].append(loss_ic.item())
        history["phys"].append(loss_phys.item())
        history["data"].append(loss_data.item())
        history["lr"].append(optimizer.param_groups[0]["lr"])

        if epoch % print_every == 0:
            tqdm.write(f"Epoch {epoch:6d} | Loss: {loss.item():.3e} | "
                       f"IC: {loss_ic.item():.3e} | Phys: {loss_phys.item():.3e} | "
                       f"Data: {loss_data.item():.3e} | lr: {history['lr'][-1]:.2e}")

    return history

In [ ]:
def compute_pinn_experiment(
    p,
    hidden_dim,
    num_layers,
    N_f,
    N_data,
    noise_std,
    epochs,
    lr,
    lambda_ic,
    lambda_phys,
    lambda_data,
    seed,
    device,
):
    """
    Complete pipeline for ONE configuration:
      1. synthetic measurements      2. normaliser
      3. collocation points          4. fresh network + training
      5. validation against the high-accuracy reference solution

    Returns
    -------
    results : dict
        Model, loss history, test grid, PINN prediction, reference solution,
        measurements and error metrics.
    """
    # -------------------------------------------------------------------------
    # STEP 1: SYNTHETIC MEASUREMENTS
    # -------------------------------------------------------------------------
    t_data_np, theta_data_np = generate_experimental_data(p, N_data, noise_std, seed)

    # -------------------------------------------------------------------------
    # STEP 2: NORMALISER (statistics taken from the measured data only)
    # -------------------------------------------------------------------------
    norm = Normaliser(p.t_start, p.t_end, theta_data_np, device)

    # NumPy (float64) -> torch (float32) column tensors on the compute device
    t_data     = torch.tensor(t_data_np, dtype=torch.float32, device=device).unsqueeze(1)
    theta_data = torch.tensor(theta_data_np, dtype=torch.float32, device=device)

    # -------------------------------------------------------------------------
    # STEP 3: COLLOCATION POINTS AND INITIAL TIME
    # -------------------------------------------------------------------------
    t_f, t_ic = generate_domain_points(p.t_start, p.t_end, N_f, device)

    # -------------------------------------------------------------------------
    # STEP 4: FRESH NETWORK AND TRAINING
    # -------------------------------------------------------------------------
    # Fixing the seed makes the random weight initialisation reproducible.
    torch.manual_seed(seed)
    model = PINN2DOF(hidden_dim, num_layers).to(device)
    n_params = sum(prm.numel() for prm in model.parameters())
    print(f"Network: 1 input -> {num_layers} x {hidden_dim} tanh -> 2 outputs "
          f"({n_params} trainable parameters)")

    t0 = time.time()
    history = train_pinn(
        model, norm, p, t_f, t_ic, t_data, theta_data,
        epochs=epochs, lr=lr,
        lambda_ic=lambda_ic, lambda_phys=lambda_phys, lambda_data=lambda_data,
        print_every=max(1, epochs // 10),
    )
    print(f"Training time: {time.time() - t0:.1f} s")

    # -------------------------------------------------------------------------
    # STEP 5: VALIDATION AGAINST THE REFERENCE SOLUTION
    # -------------------------------------------------------------------------
    # Fine test grid, independent of collocation and measurement points.
    t_test_np = np.linspace(p.t_start, p.t_end, 1000)
    t_test = torch.tensor(t_test_np, dtype=torch.float32, device=device).unsqueeze(1)

    # torch.no_grad(): no graph is built -> faster, less memory (no derivatives needed).
    with torch.no_grad():
        theta_pred = predict_theta(model, t_test, norm).cpu().numpy()   # (N, 2)

    theta_ref = solve_reference(p, t_test_np)[:, 0:2]                   # (N, 2)

    abs_err = np.abs(theta_pred - theta_ref)
    max_err = abs_err.max(axis=0)                                       # per angle
    rmse    = np.sqrt(np.mean((theta_pred - theta_ref)**2, axis=0))     # per angle
    # Relative L2 error: scale-independent measure, common in the PINN literature
    rel_l2  = (np.linalg.norm(theta_pred - theta_ref, axis=0)
               / np.linalg.norm(theta_ref, axis=0))

    print("\nValidation against reference solution (1000 test points):")
    for k, name in enumerate(["theta1", "theta2"]):
        print(f"  {name}: max |error| = {max_err[k]:.3e} rad | "
              f"RMSE = {rmse[k]:.3e} rad | rel. L2 = {rel_l2[k]:.3e}")

    return {
        "model": model,
        "history": history,
        "t_test": t_test_np,
        "theta_pred": theta_pred,
        "theta_ref": theta_ref,
        "t_data": t_data_np,
        "theta_data": theta_data_np,
        "abs_err": abs_err,
        "max_err": max_err,
        "rmse": rmse,
        "rel_l2": rel_l2,
    }

In [ ]:
def plot_pinn_results(results, title):
    """
    2 x 2 dashboard:
      (a) theta1(t): PINN vs reference, with measurements
      (b) theta2(t): PINN vs reference, with measurements
      (c) training loss components (log scale)
      (d) absolute error of both angles vs time (log scale)
    """
    t      = results["t_test"]
    pred   = results["theta_pred"]
    ref    = results["theta_ref"]
    t_d    = results["t_data"]
    th_d   = results["theta_data"]
    hist   = results["history"]

    fig, axes = plt.subplots(2, 2, figsize=(15, 9))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    # =========================================================================
    # PANELS (a), (b): ANGLES OF BOTH ARMS
    # =========================================================================
    for k, (ax, color, arm) in enumerate(zip(axes[0], ["tab:blue", "tab:red"],
                                             ["Top arm", "Bottom arm"])):
        ax.scatter(t_d, th_d[:, k], s=10, color=color, alpha=0.35, label="Noisy measurements")
        ax.plot(t, ref[:, k], color=color, linewidth=2.5, alpha=0.6, label="Reference (DOP853)")
        ax.plot(t, pred[:, k], "k--", linewidth=1.6, label="PINN")
        ax.set_title(f"({'ab'[k]}) {arm}: $\\theta_{k+1}(t)$", fontsize=12, fontweight="bold")
        ax.set_xlabel("Time $t$ [s]")
        ax.set_ylabel(f"$\\theta_{k+1}$ [rad]")
        ax.legend(fontsize=9)
        ax.grid(True, linestyle=":", alpha=0.6)

    # =========================================================================
    # PANEL (c): LOSS HISTORY
    # =========================================================================
    ax = axes[1, 0]
    ax.semilogy(hist["total"], color="black", linewidth=1.4, label="Total")
    ax.semilogy(hist["phys"],  color="tab:red",    alpha=0.8, linewidth=1.0, label="Physics $\\mathcal{L}_{phys}$")
    ax.semilogy(hist["ic"],    color="tab:green",  alpha=0.8, linewidth=1.0, label="Initial cond. $\\mathcal{L}_{IC}$")
    ax.semilogy(hist["data"],  color="tab:blue",   alpha=0.8, linewidth=1.0, label="Data $\\mathcal{L}_{data}$")
    ax.set_title("(c) Training loss components", fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss (log scale)")
    ax.legend(fontsize=9)
    ax.grid(True, which="both", linestyle=":", alpha=0.6)

    # =========================================================================
    # PANEL (d): POINTWISE ERROR VS TIME
    # =========================================================================
    ax = axes[1, 1]
    ax.semilogy(t, results["abs_err"][:, 0], color="tab:blue", label="$|\\hat\\theta_1-\\theta_1|$")
    ax.semilogy(t, results["abs_err"][:, 1], color="tab:red",  label="$|\\hat\\theta_2-\\theta_2|$")
    ax.set_title("(d) Absolute error vs reference", fontsize=12, fontweight="bold")
    ax.set_xlabel("Time $t$ [s]")
    ax.set_ylabel("Absolute error [rad]")
    ax.legend(fontsize=9)
    ax.grid(True, which="both", linestyle=":", alpha=0.6)

    plt.tight_layout()
    plt.show()

In [ ]:
def main(dd):
    """
    Main execution pipeline for the double-pendulum PINN:
    1. Selects the compute device.
    2. Defines physical parameters and hyperparameters.
    3. Runs the PINN experiment (data generation, training, validation).
    4. Plots the results dashboard.
    """
    print("=======================================================")
    print("   DOUBLE PENDULUM (2-DOF) PINN WORKFLOW")
    print("=======================================================\n")

    # -------------------------------------------------------------------------
    # STEP 1: DEVICE CONFIGURATION
    # -------------------------------------------------------------------------
    device = get_device(dd)

    # -------------------------------------------------------------------------
    # STEP 2: PHYSICAL PARAMETERS & HYPERPARAMETERS
    # -------------------------------------------------------------------------
    p = DoublePendulumParams()        # defaults: 45°, L = 1 m, m = 1 kg, t in [0, 5] s

    hidden_dim  = 64      # neurons per hidden layer
    num_layers  = 4       # hidden layers (richer dynamics than a single pendulum)
    N_f         = 1000    # collocation points for the physics residual
    N_data      = 250     # noisy measurement instants (both angles at each instant)
    noise_std   = 0.005   # sensor noise standard deviation [rad]
    epochs      = 20000   # Adam iterations (lr halved every epochs/4)
    lr          = 1e-3    # initial learning rate
    lambda_ic   = 1.0     # weight of the initial-condition loss
    lambda_phys = 0.01    # weight of the physics loss: residuals are O(g) ~ 10 rad/s^2,
                          # i.e. ~100x the normalised data term; 1.0 stalls training
    lambda_data = 1.0     # weight of the data loss (set 0.0 for a pure-physics PINN)
    seed        = 123     # reproducibility of data and weight initialisation

    # -------------------------------------------------------------------------
    # STEP 3: RUN THE PINN EXPERIMENT
    # -------------------------------------------------------------------------
    results = compute_pinn_experiment(
        p, hidden_dim, num_layers, N_f, N_data, noise_std,
        epochs, lr, lambda_ic, lambda_phys, lambda_data, seed, device,
    )

    # -------------------------------------------------------------------------
    # STEP 4: VISUALISE
    # -------------------------------------------------------------------------
    plot_pinn_results(results, title="Double pendulum – hybrid PINN (IC + physics + noisy data)")

    print("=======================================================")
    print("   PINN WORKFLOW SUCCESSFULLY COMPLETED!")
    print("=======================================================")

    return results

In [ ]:
# Standard Python entry point check: allows the code to be run as a script
# or imported as a module into another notebook.
if __name__ == "__main__":

    # Which device: CPU, Apple-Silicon GPU (MPS), CUDA GPU, or automatic (None).
    dd = ['cpu', 'mps', 'cuda', None]
    results = main(dd[3])